<a href="https://colab.research.google.com/github/MUHAMMADAFFAN786/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUHAMMADAFFAN786/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

I selected two findings from the FlyRank research paper that are useful for practicing methodology review.

### Finding 1 — Content freshness and performance

The paper reports an observed relationship between content freshness and performance.

**My methodology question:**  
How is “freshness” defined, and what is the source of the freshness label or measurement? I would want to know whether freshness is based on a recorded update date, another timestamp, or a derived category.

I would also ask whether the validation design separates observations by time. If older and newer observations are mixed together, the result may describe an association in the available data without showing that a freshness change would cause future performance to improve.

This does not invalidate the finding. It helps clarify whether the evidence supports an observed relationship or a stronger causal claim.

### Finding 2 — Reader engagement and performance

The paper also reports patterns involving reader engagement and content performance.

**My methodology question:**  
How are engagement measures such as sessions or scroll depth measured, and are these measurements available before or after the performance outcome being studied?

I would check whether any engagement variable could contain information from the same period as the outcome. If so, it may be useful for describing the data but could become leakage if it is used as a feature to predict an outcome that occurs earlier.

I would also check whether the reported relationship remains consistent across different samples or time periods rather than depending on one evaluation window.

### My review approach

These are methodology questions rather than criticisms of the findings. The purpose is to understand how the labels, measurements, comparisons, and validation design support the conclusions before treating the findings as broadly generalizable.

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

The Week-5 model used the existing evaluation setup for comparison with the baseline.

For this audit, I use a client-grouped split. Pages belonging to the same client should not be split between training and testing because that can make the test set less independent from the training data.

The original Week-5 result is reported as the “before” result. The client-holdout evaluation is the “after” result.

The purpose is not to make the score look better. The purpose is to obtain a more conservative estimate of performance on clients that were not used during training.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [37]:
import os
import json
import pandas as pd
import numpy as np

DATA_PATH = "data/raw/content_refresh_anonymized.csv"

assert os.path.exists(DATA_PATH), f"Dataset not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [38]:
print("Possible client columns:")
print([c for c in df.columns if "client" in c.lower()])

print("\nPossible time/date columns:")
print([
    c for c in df.columns
    if any(x in c.lower() for x in ["date", "time", "updated", "created"])
])

print("\nTarget-related columns:")
print([
    c for c in df.columns
    if any(x in c.lower() for x in ["trend", "label", "declin"])
])

Possible client columns:
['client_id']

Possible time/date columns:
['days_since_last_update']

Target-related columns:
['trend_direction', 'trend_pct']


In [39]:
# The FlyRank reference defines the label from trend_direction.
assert "trend_direction" in df.columns, "trend_direction column not found."

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down")
)

print("Target distribution:")
print(df["is_declining_label"].value_counts())
print("\nTarget rate:")
print(df["is_declining_label"].mean())

Target distribution:
is_declining_label
True     16262
False    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


In [40]:
client_candidates = [
    c for c in df.columns
    if "client" in c.lower()
]

print("Client candidates:", client_candidates)

if not client_candidates:
    raise ValueError(
        "No client-like column was found. Inspect the dataset columns before continuing."
    )

Client candidates: ['client_id']


In [41]:
CLIENT_COL = client_candidates[0]

print("Using client column:", CLIENT_COL)
print("Unique clients:", df[CLIENT_COL].nunique())

Using client column: client_id
Unique clients: 32


In [42]:
from sklearn.model_selection import GroupShuffleSplit

X = df.drop(columns=["is_declining_label"])
y = df["is_declining_label"]
groups = df[CLIENT_COL]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df[CLIENT_COL].nunique())
print("Test clients:", test_df[CLIENT_COL].nunique())

overlap = set(train_df[CLIENT_COL]).intersection(
    set(test_df[CLIENT_COL])
)

print("Client overlap:", len(overlap))

Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import precision_score, accuracy_score, roc_auc_score

# Use columns that are safe from direct target leakage.
leakage_columns = {
    "is_declining_label",
    "trend_direction",
    "trend_pct"
}

candidate_features = [
    c for c in df.columns
    if c not in leakage_columns
    and c != CLIENT_COL
]

# Keep numeric and categorical features separate.
numeric_features = [
    c for c in candidate_features
    if pd.api.types.is_numeric_dtype(df[c])
]

categorical_features = [
    c for c in candidate_features
    if c not in numeric_features
]

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 29
Categorical features: 12


In [29]:
X_train = train_df[candidate_features]
X_test = test_df[candidate_features]

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))

Accuracy: 0.6904105143598896
ROC-AUC: 0.7503955817068774


In [30]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return np.mean(np.asarray(y_true)[order])

honest_p50 = precision_at_k(
    y_test.reset_index(drop=True),
    prob,
    k=50
)

print(f"Honest client-holdout Precision@50: {honest_p50:.3f}")

Honest client-holdout Precision@50: 0.800


In [31]:
results_path = "outputs/model_results.json"

if os.path.exists(results_path):
    with open(results_path, "r") as f:
        results = json.load(f)

    before_p50 = results["models"]["random_forest"]["precision_at_50"]

    comparison = pd.DataFrame({
        "Evaluation": [
            "Week-5 original evaluation",
            "ML-09 client-holdout evaluation"
        ],
        "Precision@50": [
            before_p50,
            honest_p50
        ]
    })

    comparison["Precision@50"] = comparison["Precision@50"].round(3)

    display(comparison)

else:
    print("outputs/model_results.json is not available.")
    print("The honest client-holdout result was measured as:",
          round(honest_p50, 3))

outputs/model_results.json is not available.
The honest client-holdout result was measured as: 0.8


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I checked the final feature set for information that would only be available after the prediction point.

The most important leakage risk in this task is the target definition itself. The target `is_declining_label` is derived from `trend_direction`, so `trend_direction` must not be used as a feature. `trend_pct` is also excluded because it is directly related to the same outcome information.

I also excluded the client identifier from the model features because it identifies the group rather than representing a generalizable predictive signal.

The validation split was checked for client overlap. A client should appear in either the training set or the test set, not both.

Based on these checks, I did not identify an intentional direct target leak in the final feature set. However, feature availability and temporal relationships remain important limitations, so the results should be treated as measured evidence rather than proof of generalization.

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [32]:
print("=== Leakage checks ===")

print(
    "Target included in features:",
    "is_declining_label" in candidate_features
)

print(
    "trend_direction included:",
    "trend_direction" in candidate_features
)

print(
    "trend_pct included:",
    "trend_pct" in candidate_features
)

print(
    "Client column included:",
    CLIENT_COL in candidate_features
)

print(
    "Train/test client overlap:",
    len(overlap)
)

=== Leakage checks ===
Target included in features: False
trend_direction included: False
trend_pct included: False
Client column included: False
Train/test client overlap: 0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The Random Forest model performs better than the baseline and can generalize well to new data.

### Safer claim

The Random Forest model showed measurable Precision@50 performance under the evaluated split. The client-holdout result provides a more conservative estimate of performance on unseen clients than a row-level split. The result is directional evidence that the selected features contain useful predictive signal, but it does not by itself prove broad real-world generalization.

Additional testing on future observations and different client samples would be needed before making a stronger claim.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 5. Self-check

- [x] I reviewed two research-paper findings constructively.
- [x] I asked methodology questions about labels, measurements, and validation.
- [x] I compared the Week-5 evaluation with a client-grouped evaluation.
- [x] I checked that training and test clients do not overlap.
- [x] I checked the feature set for direct target leakage.
- [x] I excluded `trend_direction` and `trend_pct` from model features.
- [x] I inspected the model's measured performance rather than inventing a score.
- [x] I rewrote broad claims using safer language.
- [x] I used words such as observed, measured, directional, and decision-support.
- [x] I will commit the executed notebook to `work/notebooks/w06_validation_audit.ipynb`.